# AI-Driven Student Performance Prediction System

This notebook reproduces the complete machine learning workflow for the student pass/fail prediction system. It is designed for project review, portfolio demonstration, and academic submission.

## Workflow Architecture

```mermaid
flowchart TD
    A[Dataset] --> B[Data Cleaning]
    B --> C[Preprocessing]
    C --> D[EDA]
    D --> E[Feature Engineering]
    E --> F[Train/Test Split]
    F --> G[Model Comparison]
    G --> H[Best Model]
    H --> I[Prediction]
    I --> J[Streamlit Dashboard]
```


## 1. Environment Setup

Place `student-mat.csv` and/or `student-por.csv` in `../backend/data/raw/` before running this notebook.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from backend.config.config import FIGURES_DIR, PROCESSED_DATA_DIR, REPORTS_DIR, ensure_directories
from backend.services.data_preprocessing import clean_student_data, load_student_data, split_features_target
from backend.services.eda import run_eda
from backend.services.predict import predict_student
from backend.models.train import train_and_select_model

ensure_directories()

## 2. Load Dataset

The loader automatically combines Mathematics and Portuguese course files when both are available.

In [ ]:
raw_df = load_student_data()
raw_df.head()

In [ ]:
raw_df.shape, raw_df.columns.tolist()

## 3. Data Cleaning and Feature Engineering

The final grade `G3` is converted into a binary target. `G3` is then excluded from features in the training pipeline to avoid target leakage.

In [ ]:
cleaned_df = clean_student_data(raw_df)
cleaned_df.to_csv(PROCESSED_DATA_DIR / 'student_performance_processed.csv', index=False)
cleaned_df.head()

In [ ]:
cleaned_df['pass'].value_counts().rename({0: 'Fail', 1: 'Pass'})

## 4. Exploratory Data Analysis

EDA figures are generated automatically and saved to `../backend/outputs/figures/`.

In [ ]:
run_eda(cleaned_df)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

In [ ]:
cleaned_df.describe().T

## 5. Feature and Target Split

In [ ]:
X, y = split_features_target(cleaned_df)
print(f'Feature matrix: {X.shape}')
print(f'Target vector: {y.shape}')
X.head()

## 6. Model Comparison and Evaluation

The training pipeline compares Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and Support Vector Machine. The best model is selected using ROC-AUC, with accuracy as a tie-breaker.

In [ ]:
best_model, comparison_df = train_and_select_model()
comparison_df

## 7. Generated Evaluation Artifacts

In [ ]:
pd.read_csv(REPORTS_DIR / 'model_comparison.csv')

In [ ]:
print((REPORTS_DIR / 'classification_report.txt').read_text())

Expected visual outputs:

- `correlation_heatmap.png`
- `feature_importance.png`
- `confusion_matrix.png`
- `roc_curve.png`
- `precision_recall_curve.png`
- `class_distribution.png`
- `model_accuracy_comparison.png`

## 8. Single Student Prediction

In [ ]:
sample_student = {
    'subject': 'mathematics',
    'school': 'GP',
    'sex': 'F',
    'age': 17,
    'address': 'U',
    'famsize': 'GT3',
    'Pstatus': 'T',
    'Medu': 4,
    'Fedu': 4,
    'Mjob': 'teacher',
    'Fjob': 'services',
    'reason': 'course',
    'guardian': 'mother',
    'traveltime': 1,
    'studytime': 2,
    'failures': 0,
    'schoolsup': 'yes',
    'famsup': 'no',
    'paid': 'no',
    'activities': 'yes',
    'nursery': 'yes',
    'higher': 'yes',
    'internet': 'yes',
    'romantic': 'no',
    'famrel': 4,
    'freetime': 3,
    'goout': 3,
    'Dalc': 1,
    'Walc': 1,
    'health': 4,
    'absences': 2,
    'G1': 15,
    'G2': 14,
}

predict_student(sample_student)

## 9. Summary

This notebook demonstrates a complete applied ML workflow: data ingestion, cleaning, EDA, preprocessing, model comparison, evaluation, artifact generation, and prediction. The saved model can be used by both the command-line script and Streamlit dashboard.